# 1. Initialisation

In [25]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# 2. La fonction search

In [26]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

# 3. Définir le Tool

In [27]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"]
        }
    }
}

# 4. Question utilisateur

In [34]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant. When a relevant tool is available, use it instead of answering from your own knowledge."
    },
    {
        "role": "user",
        "content": "I just discovered the course. Can I join it?"
    }
]

# 5. Premier appel du LLM

In [35]:
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto",
    temperature=0
)

 # 6. Vérifier si le LLM veut appeler une fonction

In [36]:
message = response.choices[0].message

print(message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='4fe2w6n02', function=Function(arguments='{"query":"joining the course after discovery"}', name='search'), type='function')]


# 7. Exécuter la fonction

ouvrir la base faq.db

In [38]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [39]:
import json

tool_call = message.tool_calls[0]

args = json.loads(tool_call.function.arguments)

results = search(**args)

result_json = json.dumps(results, indent=2)

# 8. Ajouter les nouveaux messages

In [40]:
messages.append({
    "role": "assistant",
    "content": message.content,
    "tool_calls": message.tool_calls
})

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

# 9. Deuxième appel

In [41]:
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool]
)

# 10. Réponse finale

In [43]:
print(response.choices[0].message.content)

Yes, you can join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
